In [ ]:
!pip install ultralytics kaggle opencv-python-headless -q
import os
import torch
from ultralytics import YOLO
from google.colab import drive, userdata

drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/MultiCamera_Vehicle_ReIdentification'
DATASET_DIR = '/content/dataset'
RUNS_DIR = os.path.join(PROJECT_ROOT, 'yolo_runs')

In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

!kaggle datasets download -d fareselmenshawii/large-license-plate-dataset -p /content
!unzip -q -o /content/large-license-plate-dataset.zip -d {DATASET_DIR}

yaml_content = f"""
path: {DATASET_DIR}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['license_plate']
"""
with open(os.path.join(DATASET_DIR, 'data.yaml'), 'w') as f:
    f.write(yaml_content)
print("✅ Dataset and config ready for training!")

In [ ]:
# Define the specific run name and paths
RUN_NAME = 'license_plate_detector'
RUN_PATH = os.path.join(RUNS_DIR, RUN_NAME)
LAST_CHECKPOINT = os.path.join(RUN_PATH, 'weights', 'last.pt')

# Set epochs (e.g., 50 for your main run)
TOTAL_EPOCHS = 50

print("🔍 Checking for existing training checkpoints...")

if os.path.exists(LAST_CHECKPOINT):
    # --- RESUME INTERRUPTED TRAINING ---
    print(f"🔄 Interrupted training found! Resuming from {LAST_CHECKPOINT}...")
    model = YOLO(LAST_CHECKPOINT)
    results = model.train(resume=True)
else:
    # --- START FRESH TRAINING ---
    print("🚀 No previous checkpoint found. Starting fresh training run...")
    model = YOLO('yolov8n.pt')

    results = model.train(
        data=os.path.join(DATASET_DIR, 'data.yaml'),
        epochs=TOTAL_EPOCHS,
        imgsz=640,
        project=RUNS_DIR,
        name=RUN_NAME,
        save=True,
        exist_ok=True  # 👈 THIS PREVENTS YOLO FROM CREATING FOLDERS WITH NUMBERS
    )

print("✅ Training complete! Best weights saved to Drive.")